# Module 1 — Corpus Preparation

This notebook cleans the raw Bangla-English corpus while preserving Bengali characters and sentiment negations. It also creates the one fixed 70/15/15 split used by every later module.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from collections import Counter
from src.config import RAW_DATA_PATH
from src.preprocessing import prepare_corpus, preprocess_text, calculate_edit_distance

## Inspect the unchanged raw dataset

In [ ]:
raw_data = pd.read_csv(RAW_DATA_PATH)
print('Shape:', raw_data.shape)
print('Columns:', raw_data.columns.tolist())
print('\nData types:\n', raw_data.dtypes)
print('\nMissing values:\n', raw_data.isna().sum())
print('\nExact duplicate rows:', raw_data.duplicated().sum())
print('\nLabel counts:\n', raw_data['Label'].value_counts().sort_index())
raw_data.head(10)

## Check the most important cleaning rules

In [ ]:
examples = [
    'Movie ta REALLY bhalo!!! https://example.com',
    'movie ta bhalo na',
    'সার্ভিস ভালো না',
]
for example in examples:
    print(example, '->', preprocess_text(example))

print('Edit distance (bhalo, vhalo):', calculate_edit_distance('bhalo', 'vhalo'))

## Process the complete corpus
Stop-word removal and English lemmatization remain disabled because that is the safest default for code-mixed sentiment text.

In [ ]:
processed_data = prepare_corpus(remove_stopwords=False, lemmatize_english=False)
processed_data.head(10)

## Review saved examples and corpus statistics

In [ ]:
display(processed_data.groupby('label', sort=False).head(3)[['original_text', 'processed_text', 'label']])
display(processed_data['token_count'].describe(percentiles=[0.90, 0.95]))
display(processed_data['label'].value_counts())
token_counts = Counter(token for text in processed_data['processed_text'] for token in text.split())
display(pd.DataFrame(token_counts.most_common(20), columns=['token', 'frequency']))